# VSPAERO mesh parameter sweep / representation VV

このNotebookは、**基準 `.vsp3` を一度だけ physical mesh equalization し、その equalized baseline から独立した1変数 mesh caseを作り、各caseを `ThinWing / ThickWing / Hybrid / ThickAll` の4 representationでVSPAERO解析する**ための実行Notebookである。

処理順は次のとおり。

1. `equalize_vspaero_tessellation()` で基準meshを作る。
2. equalized baseline から毎回独立した mesh case を作る。
3. 指定した `Tess_W` または1つの `SectTess_U` だけを変更する。
4. 同じ mesh case を4 representationへ分岐する。
5. `vsp_sweep()` で Thick/Thin set を明示してVSPAEROを実行する。
6. `analyze_vspaero_mesh_quality()` で mesh / $C_p$ / Kutta / LOD / surface-wake consistency を同じ方法で診断する。
7. 1 case × 1 representation をCSVの1行として保存する。

`equalize_vspaero_tessellation()` は manual parameter sweep の各case内では再実行しない。equalization を再実行すると複数の mesh parameter が同時に変わり、1変数DOEではなくなるためである。

このNotebookは solver や mesh-quality parser を再実装しない。

## 1. import とリポジトリ位置

In [10]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import shutil
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import openvsp as vsp
from IPython.display import display

start_dir = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in [start_dir, *start_dir.parents]
        if (candidate / "src" / "AnalysisVSPAERO.py").exists()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root containing src/AnalysisVSPAERO.py. "
        "Run this notebook inside the repository or set repo_root manually."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.AnalysisVSPAERO import vsp_sweep  # noqa: E402
from src.VSPAEROMesh import equalize_vspaero_tessellation  # noqa: E402
from src.VSPAEROMeshQuality import analyze_vspaero_mesh_quality  # noqa: E402
from src.util import find_geom_parm, find_one_geom, get_xsec_parm_id, workdir  # noqa: E402

print("repo_root:", repo_root)

repo_root: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP


## 2. 基準モデル・飛行条件・equalization条件

通常変更する設定はこのセルに集約する。

`reference_tess_w` は reference Wing の chordwise resolution を定義する。`equalize_vspaero_tessellation()` はその実3次元 W-edge median を target mesh size とし、`SectTess_U / Tess_U / Tess_W` と active Wing End Cap の `CapUMinTess` を調整する。

既定値は Boeing 777-9X モデル用である。別モデルを使う場合は `model_name` と `reference_wing_name` を変更する。

In [11]:
# model_name = "Boeing_777-9x_mod"
# reference_wing_name = "WingGeom"
# alpha_deg = 3.0
# mach = 0.6
# reynolds = 1.0e7

model_name = "CRM-HS"
reference_wing_name = "WingGeom"
alpha_deg = 3.0
mach = 0.6
reynolds = 1.0e7

# model_name = "G103A"
# reference_wing_name = "WingGeom"
# alpha_deg = 2.0
# mach = 0.0809848
# reynolds = 5.0e6

# model_name = "SampleGlider"
# reference_wing_name = "WingGeom"
# alpha_deg = 0.0
# mach = 0.027
# reynolds = 5.0e6

reference_tess_w = 17

base_vsp3_path = (
    repo_root
    / "examples"
    / "models"
    / model_name
    / f"{model_name}.vsp3"
)

output_dir = (
    repo_root
    / "examples"
    / "notebooks"
    / "mesh_parameter_sweep"
    / f"results.{model_name}"
)
results_path = output_dir / f"mesh_parameter_sweep.{model_name}.csv"

# None means all adjustable surface Geoms, including the reference Wing.
equalize_geom_names = None
equalize_tolerance = 0.10
equalize_max_iterations = 4

ncpu = 8

fixed_wake_flag = True
wake_num_iter = None
wake_num_nodes = 32
verbose = 0

output_dir.mkdir(parents=True, exist_ok=True)
if not base_vsp3_path.exists():
    raise FileNotFoundError(base_vsp3_path)

print("base_vsp3_path:", base_vsp3_path)
print("output_dir:", output_dir)
print("reference wing / Tess_W:", reference_wing_name, reference_tess_w)
print("flight condition:", alpha_deg, mach, reynolds)
print(
    "wake settings:",
    f"FixedWakeFlag={fixed_wake_flag}",
    f"WakeNumIter={wake_num_iter}",
    f"NumWakeNodes={wake_num_nodes}",
)

base_vsp3_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\models\CRM-HS\CRM-HS.vsp3
output_dir: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\mesh_parameter_sweep\results.CRM-HS
reference wing / Tess_W: WingGeom 17
flight condition: 3.0 0.6 10000000.0
wake settings: FixedWakeFlag=True WakeNumIter=None NumWakeNodes=32


## 3. equalized baseline を作る

この処理はNotebook全体で1回だけ行う。結果CSVと equalized `.vsp3` を `_equalized_baseline` に保存し、以後の全caseはこのファイルから作る。

In [12]:
equalized_dir = output_dir / "_equalized_baseline"
if equalized_dir.exists():
    # shutil.rmtree(equalized_dir)
    pass
else:
    equalized_dir.mkdir(parents=True)

equalized_vsp3_path = equalized_dir / f"{model_name}.equalized.vsp3"
equalization_report_path = equalized_dir / "equalize_vspaero_tessellation.csv"

equalization_report = equalize_vspaero_tessellation(
    input_vsp3_path=base_vsp3_path,
    output_vsp3_path=equalized_vsp3_path,
    reference_wing_name=reference_wing_name,
    reference_tess_w=reference_tess_w,
    geom_names=equalize_geom_names,
    tolerance=equalize_tolerance,
    max_iterations=equalize_max_iterations,
)
equalization_report.to_csv(equalization_report_path, index=False)

with equalized_vsp3_path.open("rb") as file_obj:
    equalized_base_vsp3_sha256 = hashlib.file_digest(file_obj, "sha256").hexdigest()

display(equalization_report)
print("equalized_vsp3_path:", equalized_vsp3_path)
print("equalized_base_vsp3_sha256:", equalized_base_vsp3_sha256)

,geom_name,geom_type,status,reason,u_tessellation_mode,target_edge_size,geometry_w_surface_length,geometry_u_surface_lengths,geometry_cap_min_surface_length,geometry_cap_max_surface_length,...,before_u_edge_median,after_u_edge_median,before_w_edge_median,after_w_edge_median,before_edge_aspect_ratio,after_edge_aspect_ratio,u_target_ratio,w_target_ratio,cap_target_ratio,surface_count
0,FuselageGeom,Fuselage,adjusted,,SectTess_U,0.770564,8.536961,"[1.616668282904786, 0.7491378698088245, 9.0851...",NaN,NaN,...,0.790141,0.812488,0.596595,0.596648,1.324416,1.361756,1.054407,0.774299,NaN,1
1,WingGeom,Wing,adjusted,reference wing,SectTess_U,0.770564,10.344246,"[11.91652605236106, 10.429786767434821, 7.3542...",0.760951,0.137777,...,0.752460,0.764460,0.770564,0.760455,1.024060,1.005267,0.992078,0.986880,0.668919,1
2,WingBodyFairing,Fuselage,adjusted,,SectTess_U,0.770564,16.456304,"[6.762563939507258, 6.25, 6.25, 6.641761579147...",NaN,NaN,...,0.781250,0.781250,0.661417,0.661417,1.181176,1.181176,1.013867,0.858354,NaN,1
3,HTailGeom,Wing,adjusted,,SectTess_U,0.770564,8.745343,[12.913938912588408],0.269849,0.119906,...,0.748081,0.748081,0.688800,0.688800,1.086064,1.086064,0.970823,0.893891,0.260165,1
4,VTailGeom,Wing,adjusted,,SectTess_U,0.770564,11.650868,[13.794819102483448],0.501238,0.136083,...,0.753391,0.753391,0.681297,0.681297,1.105818,1.105818,0.977713,0.884154,0.430285,1


equalized_vsp3_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\mesh_parameter_sweep\results.CRM-HS\_equalized_baseline\CRM-HS.equalized.vsp3
equalized_base_vsp3_sha256: 47849b083c0be3f03b7a66733fb6023fea5e879384ac1bbf5e192d0e2c804189


## 4. manual mesh sweep 定義

各phaseは equalized baseline から開始し、**指定した1つのparameterだけ**を変更する。

既定では Boeing の body-side sensitivity を切り分けるため、`fuselage` と `base fuselage` の `Tess_W` を別々に振る。equalized baseline の値は baseline case で1回だけ実行するため、`values` には含めない。

`SectTess_U` を調べる場合は1 sectionずつ追加する。

In [13]:
sweeps = [
    # {
    #     "name": "fuselage_tess_w",
    #     "geom_name": "fuselage",
    #     "parameter": "Tess_W",
    #     "values": [21, 25, 29, 33],
    # },
    # {
    #     "name": "base_fuselage_tess_w",
    #     "geom_name": "base fuselage",
    #     "parameter": "Tess_W",
    #     "values": [21, 25, 29, 33],
    # },
]

# SectTess_U の例。必要なsectionだけ有効化する。
# sweeps.append(
#     {
#         "name": "fuselage_section4_u",
#         "geom_name": "fuselage",
#         "parameter": "SectTess_U",
#         "section_index": 4,
#         "values": [12, 16, 20, 24],
#     }
# )

for sweep in sweeps:
    print(sweep)

## 5. mesh provenance と4 representation を確認

`read_geom_mesh_settings()` は baseline / 変更後 / save-reload 後を同じ形で比較するために使う。

4 representation は次の set 定義を使う。

| representation | Thin Set | Thick Set |
|---|---|---|
| `ThinWing` | `ThinGeom` | None |
| `ThickWing` | None | `ThinGeom` |
| `Hybrid` | `ThinGeom` | `ThickGeom` |
| `ThickAll` | None | `ThickAll` |

`None` は OpenVSP の `SET_NONE` を明示する。named set が存在しない、または member が0件の場合は、その representation を明示的に `skipped` とする。set membership 自体はNotebookから変更しない。

In [14]:
def read_geom_mesh_settings(geom_name: str) -> dict[str, object]:
    geom_id = find_one_geom(vsp, geom_name)
    tess_w_id = find_geom_parm(vsp, geom_id, ("Tess_W",), ("Shape",))
    if not tess_w_id:
        raise KeyError(f"Geom '{geom_name}' has no Tess_W parameter.")

    settings: dict[str, object] = {
        "geom_name": geom_name,
        "Tess_W": int(round(vsp.GetParmVal(tess_w_id))),
        "SectTess_U": [],
    }

    if int(vsp.GetNumXSecSurfs(geom_id)) == 1:
        xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)
        for xsec_index in range(1, int(vsp.GetNumXSec(xsec_surf_id))):
            xsec_id = vsp.GetXSec(xsec_surf_id, xsec_index)
            try:
                parm_id, _ = get_xsec_parm_id(vsp, xsec_id, ("SectTess_U",))
            except KeyError:
                settings["SectTess_U"] = []
                break
            settings["SectTess_U"].append(int(round(vsp.GetParmVal(parm_id))))

    return settings


vsp.ClearVSPModel()
vsp.Update()
vsp.ReadVSPFile(str(equalized_vsp3_path))
vsp.Update()

compgeom_name = "VSPAEROComputeGeometry"
vsp.SetAnalysisInputDefaults(compgeom_name)
compgeom_inputs = set(vsp.GetAnalysisInputNames(compgeom_name))
source_saved_geom_set = int(vsp.GetIntAnalysisInput(compgeom_name, "GeomSet")[0])
source_saved_thin_geom_set = (
    int(vsp.GetIntAnalysisInput(compgeom_name, "ThinGeomSet")[0])
    if "ThinGeomSet" in compgeom_inputs
    else None
)
set_none = int(vsp.SET_NONE)

tracked_geom_names = list(dict.fromkeys(sweep["geom_name"] for sweep in sweeps))
baseline_mesh_settings = {
    geom_name: read_geom_mesh_settings(geom_name)
    for geom_name in tracked_geom_names
}

representations = [
    {"name": "ThinWing", "thin_set_name": "ThinGeom", "thick_set_name": None},
    {"name": "ThickWing", "thin_set_name": None, "thick_set_name": "ThinGeom"},
    {"name": "Hybrid", "thin_set_name": "ThinGeom", "thick_set_name": "ThickGeom"},
    {"name": "ThickAll", "thin_set_name": None, "thick_set_name": "ThickAll"},
]

for representation in representations:
    skip_reasons = []
    for side in ("thin", "thick"):
        set_name = representation[f"{side}_set_name"]
        if set_name is None:
            representation[f"{side}_set_index"] = set_none
            representation[f"{side}_set_members"] = []
            continue

        set_index = int(vsp.GetSetIndex(set_name))
        if set_index == set_none:
            representation[f"{side}_set_index"] = None
            representation[f"{side}_set_members"] = []
            skip_reasons.append(f"missing Geom Set: {set_name}")
            continue

        members = [
            str(vsp.GetGeomName(geom_id))
            for geom_id in vsp.GetGeomSetAtIndex(set_index)
        ]
        representation[f"{side}_set_index"] = set_index
        representation[f"{side}_set_members"] = members
        if not members:
            skip_reasons.append(f"empty Geom Set: {set_name}")

    representation["status"] = "skipped" if skip_reasons else "ready"
    representation["skip_reason"] = "; ".join(skip_reasons)

print("saved GeomSet / ThinGeomSet in equalized source:", source_saved_geom_set, source_saved_thin_geom_set)
display(pd.DataFrame(baseline_mesh_settings.values()))
display(pd.DataFrame(representations))

saved GeomSet / ThinGeomSet in equalized source: 4 3


""


,name,thin_set_name,thick_set_name,thin_set_index,thin_set_members,thick_set_index,thick_set_members,status,skip_reason
0,ThinWing,ThinGeom,NaN,3,"[WingGeom, HTailGeom, VTailGeom]",-1,[],ready,
1,ThickWing,NaN,ThinGeom,-1,[],3,"[WingGeom, HTailGeom, VTailGeom]",ready,
2,Hybrid,ThinGeom,ThickGeom,3,"[WingGeom, HTailGeom, VTailGeom]",4,"[FuselageGeom, WingBodyFairing]",ready,
3,ThickAll,NaN,ThickAll,-1,[],5,"[FuselageGeom, WingGeom, WingBodyFairing, HTai...",ready,


## 6. 実行 mesh case の作成

In [15]:
cases: list[dict[str, object]] = [
    {
        "case_name": "baseline",
        "sweep_name": "baseline",
        "geom_name": "",
        "parameter": "baseline",
        "section_index": None,
        "requested_value": None,
    }
]

for sweep in sweeps:
    for value in sweep["values"]:
        section_index = sweep.get("section_index")
        if sweep["parameter"] == "Tess_W":
            suffix = f"w{int(value):03d}"
        elif sweep["parameter"] == "SectTess_U":
            if section_index is None:
                raise ValueError(f"{sweep['name']} requires section_index.")
            suffix = f"u{int(section_index):02d}_{int(value):03d}"
        else:
            raise ValueError(f"Unsupported mesh parameter: {sweep['parameter']}")

        cases.append(
            {
                "case_name": f"{sweep['name']}_{suffix}",
                "sweep_name": sweep["name"],
                "geom_name": sweep["geom_name"],
                "parameter": sweep["parameter"],
                "section_index": section_index,
                "requested_value": int(value),
            }
        )

display(pd.DataFrame(cases))

,case_name,sweep_name,geom_name,parameter,section_index,requested_value
0,baseline,baseline,,baseline,None,None


## 7. mesh case × representation を実行

各 mesh case は equalized baseline から作り直す。manual change 後に非対象 mesh setting が変わっていないことと、`.vsp3` save-reload 後も同じ設定であることを確認する。

同じ mesh case を representation ごとのサブディレクトリへコピーし、一意な basename で VSPAERO を実行する。`analyze_vspaero_mesh_quality()` にはすべての sidecar を明示パスで渡す。

In [16]:
rows: list[dict[str, object]] = []
sweep_start = time.perf_counter()

for case_index, case in enumerate(cases, start=1):
    case_name = str(case["case_name"])
    case_dir = output_dir / case_name

    if case_dir.exists():
        # shutil.rmtree(case_dir)
        pass
    else:
        case_dir.mkdir(parents=True)

    print(f"[{case_index}/{len(cases)}] {case_name}", flush=True)

    # Every manual mesh case starts from the same equalized baseline.
    vsp.ClearVSPModel()
    vsp.Update()
    vsp.ReadVSPFile(str(equalized_vsp3_path))
    vsp.Update()

    effective_value = None
    if case["parameter"] != "baseline":
        geom_name = str(case["geom_name"])
        geom_id = find_one_geom(vsp, geom_name)
        requested_value = int(case["requested_value"])

        if case["parameter"] == "Tess_W":
            parm_id = find_geom_parm(vsp, geom_id, ("Tess_W",), ("Shape",))
            if not parm_id:
                raise KeyError(f"Geom '{geom_name}' has no Tess_W parameter.")
            vsp.SetParmVal(parm_id, float(requested_value))
            vsp.Update()
            effective_value = int(round(vsp.GetParmVal(parm_id)))

        elif case["parameter"] == "SectTess_U":
            section_index = int(case["section_index"])
            if int(vsp.GetNumXSecSurfs(geom_id)) != 1:
                raise ValueError(f"Geom '{geom_name}' does not have exactly one XSecSurf.")
            xsec_surf_id = vsp.GetXSecSurf(geom_id, 0)
            n_xsec = int(vsp.GetNumXSec(xsec_surf_id))
            if not 1 <= section_index < n_xsec:
                raise IndexError(
                    f"section_index={section_index} is outside 1..{n_xsec - 1} "
                    f"for Geom '{geom_name}'."
                )
            xsec_id = vsp.GetXSec(xsec_surf_id, section_index)
            parm_id, _ = get_xsec_parm_id(vsp, xsec_id, ("SectTess_U",))
            vsp.SetParmVal(parm_id, float(requested_value))
            vsp.Update()
            effective_value = int(round(vsp.GetParmVal(parm_id)))

    current_mesh_settings = {
        geom_name: read_geom_mesh_settings(geom_name)
        for geom_name in tracked_geom_names
    }

    # Controlled DOE guard: only the requested mesh value may differ from the
    # equalized baseline among the tracked Geoms.
    if case["parameter"] == "baseline":
        if current_mesh_settings != baseline_mesh_settings:
            raise RuntimeError(f"{case_name}: baseline mesh settings changed unexpectedly.")
    else:
        for geom_name, settings in current_mesh_settings.items():
            baseline_settings = baseline_mesh_settings[geom_name]
            if geom_name != case["geom_name"]:
                if settings != baseline_settings:
                    raise RuntimeError(
                        f"{case_name}: non-target Geom '{geom_name}' changed; "
                        f"baseline={baseline_settings}, current={settings}"
                    )
                continue

            if case["parameter"] == "Tess_W":
                if settings["Tess_W"] != effective_value:
                    raise RuntimeError(f"{case_name}: effective Tess_W mismatch.")
                if settings["SectTess_U"] != baseline_settings["SectTess_U"]:
                    raise RuntimeError(f"{case_name}: SectTess_U changed during Tess_W sweep.")
            elif case["parameter"] == "SectTess_U":
                if settings["Tess_W"] != baseline_settings["Tess_W"]:
                    raise RuntimeError(f"{case_name}: Tess_W changed during SectTess_U sweep.")
                expected_u = list(baseline_settings["SectTess_U"])
                expected_u[int(case["section_index"]) - 1] = effective_value
                if settings["SectTess_U"] != expected_u:
                    raise RuntimeError(
                        f"{case_name}: more than one SectTess_U value changed; "
                        f"expected={expected_u}, current={settings['SectTess_U']}"
                    )

    mesh_case_vsp3_path = case_dir / f"{case_name}.mesh.vsp3"
    vsp.WriteVSPFile(str(mesh_case_vsp3_path), vsp.SET_ALL)
    vsp.Update()

    # Reload the exact mesh case written to disk before branching into
    # representations.  This catches save/reload changes in mesh parameters.
    vsp.ClearVSPModel()
    vsp.Update()
    vsp.ReadVSPFile(str(mesh_case_vsp3_path))
    vsp.Update()
    reloaded_mesh_settings = {
        geom_name: read_geom_mesh_settings(geom_name)
        for geom_name in tracked_geom_names
    }
    if reloaded_mesh_settings != current_mesh_settings:
        raise RuntimeError(
            f"{case_name}: mesh settings changed during .vsp3 save/reload; "
            f"before={current_mesh_settings}, after={reloaded_mesh_settings}"
        )
    current_mesh_settings = reloaded_mesh_settings

    with mesh_case_vsp3_path.open("rb") as file_obj:
        mesh_case_vsp3_sha256 = hashlib.file_digest(file_obj, "sha256").hexdigest()

    for representation in representations:
        representation_name = str(representation["name"])
        representation_start = time.perf_counter()

        base_row: dict[str, object] = {
            **case,
            "representation": representation_name,
            "status": representation["status"],
            "skip_reason": representation["skip_reason"],
            "effective_value": effective_value,
            "thin_set_name": representation["thin_set_name"],
            "thin_set_index": representation["thin_set_index"],
            "thin_set_members": json.dumps(representation["thin_set_members"], ensure_ascii=False),
            "thick_set_name": representation["thick_set_name"],
            "thick_set_index": representation["thick_set_index"],
            "thick_set_members": json.dumps(representation["thick_set_members"], ensure_ascii=False),
            "source_saved_geom_set": source_saved_geom_set,
            "source_saved_thin_geom_set": source_saved_thin_geom_set,
            "mesh_settings_json": json.dumps(current_mesh_settings, ensure_ascii=False),
            "equalized_base_vsp3_sha256": equalized_base_vsp3_sha256,
            "mesh_case_vsp3_sha256": mesh_case_vsp3_sha256,
        }

        if representation["status"] == "skipped":
            rows.append(base_row)
            pd.DataFrame(rows).to_csv(results_path, index=False)
            print(f"    {representation_name}: SKIPPED - {representation['skip_reason']}")
            continue

        representation_dir = case_dir / representation_name
            
        if representation_dir.exists():
            # shutil.rmtree(case_dir)
            pass
        else:
            # case_dir.mkdir(parents=True)
            representation_dir.mkdir()

        representation_vsp3_path = (
            representation_dir / f"{case_name}__{representation_name}.vsp3"
        )
        shutil.copy2(mesh_case_vsp3_path, representation_vsp3_path)

        vsp.ClearVSPModel()
        vsp.Update()
        vsp.ReadVSPFile(str(representation_vsp3_path))
        vsp.Update()

        with workdir(representation_dir):
            vsp_sweep(
                vsp=vsp,
                alpha=[alpha_deg],
                mach=[mach],
                reynolds=[reynolds],
                verbose=verbose,
                ncpu=ncpu,
                wake_num_iter=wake_num_iter,
                wake_num_nodes=wake_num_nodes,
                fixed_wake_flag=fixed_wake_flag,
                thick_geom_set=int(representation["thick_set_index"]),
                thin_geom_set=int(representation["thin_set_index"]),
            )

        for analysis_name in ("VSPAEROComputeGeometry", "VSPAEROSweep"):
            if int(vsp.GetIntAnalysisInput(analysis_name, "GeomSet")[0]) != int(
                representation["thick_set_index"]
            ):
                raise RuntimeError(
                    f"{case_name}/{representation_name}: {analysis_name} GeomSet mismatch."
                )
            if int(vsp.GetIntAnalysisInput(analysis_name, "ThinGeomSet")[0]) != int(
                representation["thin_set_index"]
            ):
                raise RuntimeError(
                    f"{case_name}/{representation_name}: {analysis_name} ThinGeomSet mismatch."
                )

        stem = representation_vsp3_path.stem
        sidecars = {
            "vsp3": representation_vsp3_path,
            "vspgeom": representation_dir / f"{stem}.vspgeom",
            "adb": representation_dir / f"{stem}.adb",
            "history": representation_dir / f"{stem}.history",
            "polar": representation_dir / f"{stem}.polar",
            "lod": representation_dir / f"{stem}.lod",
            "vspaero": representation_dir / f"{stem}.vspaero",
        }
        missing = [name for name, path in sidecars.items() if not path.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{case_name}/{representation_name}: missing VSPAERO output(s): {missing}. "
                f"Expected basename: {stem}"
            )

        diagnostics = analyze_vspaero_mesh_quality(
            adb_path=sidecars["adb"],
            vspgeom_path=sidecars["vspgeom"],
            output_dir=representation_dir / "mesh_quality",
            history_path=sidecars["history"],
            polar_path=sidecars["polar"],
            lod_path=sidecars["lod"],
            vspaero_path=sidecars["vspaero"],
            vsp3_path=sidecars["vsp3"],
        )
        summary = diagnostics["summary"]
        provenance = summary["provenance"]
        mesh = summary["mesh"]
        cp = summary["cp"]
        kutta = summary["kutta"]
        solution = summary["solution"]
        lod_summary = summary["lod"]
        coefficients = solution.get("coefficients", {})
        cp_force = cp.get("force_integral", {}) or {}

        row = {
            **base_row,
            "status": "completed",
            "elapsed_s": time.perf_counter() - representation_start,
            "representation_vsp3_sha256": (provenance.get("vsp3_file") or {}).get("sha256"),
            "vspgeom_sha256": (provenance.get("vspgeom_file") or {}).get("sha256"),
            "adb_sha256": (provenance.get("adb_file") or {}).get("sha256"),
            "history_sha256": (provenance.get("history_file") or {}).get("sha256"),
            "lod_sha256": (provenance.get("lod_file") or {}).get("sha256"),
            "n_surface_triangles": mesh.get("surface_triangle_count"),
            "n_ngons": mesh.get("ngon_count"),
            "n_extreme_mesh_cells": mesh.get("extreme_cell_count"),
            "junction_edge_min": (mesh.get("junction_edge_length") or {}).get("min"),
            "junction_edge_p10": (mesh.get("junction_edge_length") or {}).get("p10"),
            "junction_to_local_p50_ratio_min": (mesh.get("junction_to_local_p50_ratio") or {}).get("min"),
            "n_local_cp_spikes": cp.get("local_spike_count"),
            "cp_min_actual": cp.get("min_actual"),
            "kutta_checks_passed": kutta.get("checks_passed"),
            "lod_outlier_count": lod_summary.get("lod_outlier_count"),
            "lod_symmetry_warning_count": lod_summary.get("lod_symmetry_warning_count"),
            "CLi": coefficients.get("CLi"),
            "CLiw": coefficients.get("CLiw"),
            "CLtot": coefficients.get("CLtot"),
            "CLwtot": coefficients.get("CLwtot"),
            "CDi": coefficients.get("CDi"),
            "CDiw": coefficients.get("CDiw"),
            "CDtot": coefficients.get("CDtot"),
            "CDwtot": coefficients.get("CDwtot"),
            "CSi": coefficients.get("CSi"),
            "CSiw": coefficients.get("CSiw"),
            "CStot": coefficients.get("CStot"),
            "CSwtot": coefficients.get("CSwtot"),
            "CMytot": coefficients.get("CMytot"),
            "surface_wake_cl_relative_difference": solution.get("surface_wake_cl_relative_difference"),
            "surface_wake_cli_relative_difference": solution.get("surface_wake_cli_relative_difference"),
            "surface_wake_cs_absolute_difference": solution.get("surface_wake_cs_absolute_difference"),
            "CL_cp": cp_force.get("CL_cp"),
            "CMy_cp": cp_force.get("CMy_cp"),
            "L2_Residual": coefficients.get("L2_Residual"),
            "Max_Residual": coefficients.get("Max_Residual"),
        }

        cli = row.get("CLi")
        cliw = row.get("CLiw")
        csi = row.get("CSi")
        csiw = row.get("CSiw")
        row["CLi_minus_CLiw"] = (
            float(cli) - float(cliw)
            if cli is not None and cliw is not None
            else None
        )
        row["CSi_minus_CSiw"] = (
            float(csi) - float(csiw)
            if csi is not None and csiw is not None
            else None
        )

        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)
        print(
            f"    {representation_name}: triangles={row['n_surface_triangles']}, "
            f"extreme={row['n_extreme_mesh_cells']}, "
            f"CLi={row['CLi']}, CLiw={row['CLiw']}, "
            f"CSi={row['CSi']}, CSiw={row['CSiw']}",
            flush=True,
        )

results = pd.DataFrame(rows)
print(f"completed in {time.perf_counter() - sweep_start:.1f} s")
print("results_path:", results_path)


[1/1] baseline
    ThinWing: triangles=2136, extreme=0, CLi=0.494736391912, CLiw=0.499635934632, CSi=7.0292e-05, CSiw=7.4516728e-05
    ThickWing: triangles=4424, extreme=0, CLi=0.51626361131, CLiw=0.511510037112, CSi=-0.000597435711, CSiw=0.000101223943
    Hybrid: triangles=7760, extreme=6, CLi=0.371342622161, CLiw=0.351551657253, CSi=0.004085720438, CSiw=-0.001047983199
    ThickAll: triangles=9732, extreme=8, CLi=0.321604345642, CLiw=0.307789924321, CSi=0.015960823092, CSiw=-0.000230727259
completed in 100.6 s
results_path: C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP\examples\notebooks\mesh_parameter_sweep\results.CRM-HS\mesh_parameter_sweep.CRM-HS.csv


## 8. VV 結果一覧

`CLi-CLiw` と `CSi-CSiw` は diagnostic quantity として表示する。単独の validity criterion にはしない。

まず equalized baseline の4 representationを比較し、その後 manual mesh sweep の感度を見る。

In [17]:
completed = results.loc[results["status"] == "completed"].copy()

vv_columns = [
    "case_name",
    "representation",
    "parameter",
    "requested_value",
    "effective_value",
    "n_surface_triangles",
    "n_ngons",
    "n_extreme_mesh_cells",
    "junction_edge_min",
    "n_local_cp_spikes",
    "cp_min_actual",
    "kutta_checks_passed",
    "lod_outlier_count",
    "CLi",
    "CLiw",
    "CLi_minus_CLiw",
    "CL_cp",
    "CDi",
    "CDiw",
    "CSi",
    "CSiw",
    "CSi_minus_CSiw",
    "CMytot",
    "elapsed_s",
]

display(results[[
    "case_name", "representation", "status", "skip_reason",
    "thin_set_name", "thin_set_members", "thick_set_name", "thick_set_members"
]])

display(completed.loc[completed["case_name"] == "baseline", vv_columns])
display(completed[vv_columns])

,case_name,representation,status,skip_reason,thin_set_name,thin_set_members,thick_set_name,thick_set_members
0,baseline,ThinWing,completed,,ThinGeom,"[""WingGeom"", ""HTailGeom"", ""VTailGeom""]",NaN,[]
1,baseline,ThickWing,completed,,NaN,[],ThinGeom,"[""WingGeom"", ""HTailGeom"", ""VTailGeom""]"
2,baseline,Hybrid,completed,,ThinGeom,"[""WingGeom"", ""HTailGeom"", ""VTailGeom""]",ThickGeom,"[""FuselageGeom"", ""WingBodyFairing""]"
3,baseline,ThickAll,completed,,NaN,[],ThickAll,"[""FuselageGeom"", ""WingGeom"", ""WingBodyFairing""..."


,case_name,representation,parameter,requested_value,effective_value,n_surface_triangles,n_ngons,n_extreme_mesh_cells,junction_edge_min,n_local_cp_spikes,...,CLiw,CLi_minus_CLiw,CL_cp,CDi,CDiw,CSi,CSiw,CSi_minus_CSiw,CMytot,elapsed_s
0,baseline,ThinWing,baseline,None,None,2136,1068,0,NaN,0,...,0.499636,-0.004900,0.370179,0.011361,0.015085,0.000070,0.000075,-0.000004,0.346170,16.231953
1,baseline,ThickWing,baseline,None,None,4424,2224,0,NaN,0,...,0.511510,0.004754,0.388992,-0.003891,0.015722,-0.000597,0.000101,-0.000699,0.255607,30.266316
2,baseline,Hybrid,baseline,None,None,7760,3760,6,0.012952,0,...,0.351552,0.019791,0.297933,0.002067,0.006848,0.004086,-0.001048,0.005134,0.138240,24.857816
3,baseline,ThickAll,baseline,None,None,9732,4734,8,NaN,2,...,0.307790,0.013814,0.254694,-0.038380,0.005210,0.015961,-0.000231,0.016192,0.119829,28.230301


,case_name,representation,parameter,requested_value,effective_value,n_surface_triangles,n_ngons,n_extreme_mesh_cells,junction_edge_min,n_local_cp_spikes,...,CLiw,CLi_minus_CLiw,CL_cp,CDi,CDiw,CSi,CSiw,CSi_minus_CSiw,CMytot,elapsed_s
0,baseline,ThinWing,baseline,None,None,2136,1068,0,NaN,0,...,0.499636,-0.004900,0.370179,0.011361,0.015085,0.000070,0.000075,-0.000004,0.346170,16.231953
1,baseline,ThickWing,baseline,None,None,4424,2224,0,NaN,0,...,0.511510,0.004754,0.388992,-0.003891,0.015722,-0.000597,0.000101,-0.000699,0.255607,30.266316
2,baseline,Hybrid,baseline,None,None,7760,3760,6,0.012952,0,...,0.351552,0.019791,0.297933,0.002067,0.006848,0.004086,-0.001048,0.005134,0.138240,24.857816
3,baseline,ThickAll,baseline,None,None,9732,4734,8,NaN,2,...,0.307790,0.013814,0.254694,-0.038380,0.005210,0.015961,-0.000231,0.016192,0.119829,28.230301


## 9. sweep別プロット

同じ manual mesh parameter に対して、4 representation の Kutta–Joukowski / wake residual と mesh pathology がどう変わるかを見る。

convergence order や Richardson extrapolation は自動計算しない。intersection topology が変化するcaseでは、単純な nested-grid family とみなせないためである。

In [18]:
for sweep_name in completed.loc[completed["sweep_name"] != "baseline", "sweep_name"].unique():
    subset = completed.loc[completed["sweep_name"] == sweep_name]

    fig, ax = plt.subplots()
    for representation_name, group in subset.groupby("representation"):
        group = group.sort_values("effective_value")
        ax.plot(
            group["effective_value"],
            group["CLi_minus_CLiw"],
            marker="o",
            label=representation_name,
        )
    ax.axhline(0.0, linestyle="--")
    ax.set_xlabel("effective mesh parameter")
    ax.set_ylabel("CLi - CLiw")
    ax.set_title(f"{sweep_name}: global/wake lift residual")
    ax.grid(True)
    ax.legend()
    plt.show()

    fig, ax = plt.subplots()
    for representation_name, group in subset.groupby("representation"):
        group = group.sort_values("effective_value")
        ax.plot(
            group["effective_value"],
            group["CSi_minus_CSiw"],
            marker="o",
            label=representation_name,
        )
    ax.axhline(0.0, linestyle="--")
    ax.set_xlabel("effective mesh parameter")
    ax.set_ylabel("CSi - CSiw")
    ax.set_title(f"{sweep_name}: side-force residual")
    ax.grid(True)
    ax.legend()
    plt.show()

    fig, ax = plt.subplots()
    for representation_name, group in subset.groupby("representation"):
        group = group.sort_values("effective_value")
        ax.plot(
            group["effective_value"],
            group["n_extreme_mesh_cells"],
            marker="o",
            label=representation_name,
        )
    ax.set_xlabel("effective mesh parameter")
    ax.set_ylabel("extreme mesh cell count")
    ax.set_title(f"{sweep_name}: mesh pathology")
    ax.grid(True)
    ax.legend()
    plt.show()

## 10. 判定時の見方

このNotebookは1つの「最適mesh」や1つの「正しいrepresentation」を自動選択しない。

確認する主な関係は次のとおり。

- equalization 後の `CapUMinTess` と root/tip End Cap の実3次元 U-edge median。
- refinement とともに `junction_edge_min` や `n_extreme_mesh_cells` が改善するか、悪化するか。
- `CLiw` と `CL_cp` が安定しているときに `CLi` だけが変化するか。
- $\beta=0$ で `CSiw` が小さい状態を保ちながら `CSi` が増大するか。
- Kutta topology、LOD outlier、局所 $C_p$ anomaly が integrated-force change と空間的・系列的に対応するか。
- `ThinWing ↔ ThickWing` と `Hybrid ↔ ThickAll` の差が、各representation内部の surface/wake consistency と比較してどの程度大きいか。
- set membership が representation 間で異なる場合、その差を model-form comparison の provenance として残す。

manual sweep を追加する場合も、equalized baseline から開始し、一度に1 parameterまたは1 sectionだけ変更する。